# 202 · Anytime Valid Inference: E-processes

Traditional GSD requires you to pre-specify the number of looks (e.g., $K=5$). If you "peek" at the data more often, you inflate the Type I error. 

**Anytime Valid Inference (AVI)** solves this using **E-processes** (like the **mSPRT**). You can monitor the data as often as you want—even after every single sample—and the statistical guarantees remain valid. 

Instead of P-values, we use **E-values**. If the E-value exceeds $1/\alpha$, we can stop and reject the null hypothesis.

## 1. Setup & mSPRT Design

We use the `mSPRT_Johari2019` template. The key parameter here is `tau`, which represents the expected effect size you want to detect (similar to MDE).

In [ ]:
import ibis
import numpy as np
import matplotlib.pyplot as plt
from earlysign.core.ledger import Ledger
from earlysign.schema.ES3.Binomial import ArmData
from earlysign.v1.templates.mSPRT_Johari2019 import BinomialJohari2019Template

ledger = Ledger(ibis.connect("duckdb://:memory:"), "avi_events").bind(exp_id="avi_demo")
ledger.ensure()

trial = BinomialJohari2019Template(ledger)
protocol = trial.design(
    arms=["C", "T"], alpha=0.05, tau=0.02, sides="one"  # Expected difference in rates
)
trial.set_protocol(protocol)

print("mSPRT Design Initialized.")

## 2. Continuous Monitoring

We can update the results as frequently as data becomes available.

In [ ]:
p_c, p_t = 0.10, 0.15
batch_size = 100

print("Starting monitoring...")
for i in range(1, 21):  # Simulated 20 small batches
    batch = [
        ArmData(n=batch_size, success=np.random.binomial(batch_size, p_c), arm="C"),
        ArmData(n=batch_size, success=np.random.binomial(batch_size, p_t), arm="T"),
    ]

    trial.update(batch)
    report = trial.report_progress()

    # Note: AVI reports e_value instead of boundaries
    # 'status' becomes 'stop' if e_value > 1/alpha
    print(f"Batch {i}: N={report['sample_n']}, Status={report['status']}")

    if report["status"] == "STOP_EFFICACY":
        print(f">>> E-value crossed threshold! Evidence is sufficient.")
        break

## 3. Visualization

In AVI, we typically plot the **Log E-value** trajectory. If it crosses the $\log(1/\alpha)$ line, we stop.

In [ ]:
# The template provides a specialized plot for AVI trajectories
trial.report_result()
print("Final Report Generated.")

## 4. Summary

- **E-processes**: Allow for true continuous monitoring without alpha-spending "looks".
- **Peeking is Safe**: No penalty for checking the results after every user interaction.
- **Efficiency**: Often detect strong signals faster than fixed-look GSD.

Next, we will look at **Confidence Sequences**, which allow us to report "live" confidence intervals that are valid even under peeking.